# Live Parquet Batch Inspector (Extractor v8 format)
Inspect extraction checkpoints from `batch_results/results_batch_*` as they are being generated by `spotify_extractor_v8.ipynb`.

This notebook supports both `.parquet` and `.pkl` checkpoints (same behavior as extractor v8).

In [1]:
from pathlib import Path
from datetime import datetime
import re
import time

import polars as pl
import pandas as pd

# Must match spotify_extractor_v8.ipynb
BATCH_DIR = Path('batch_results')
PATTERN = 'results_batch_*'

print(f'Watching batch directory: {BATCH_DIR.resolve()}')

Watching batch directory: /home/build/martin/spotify_v2/batch_results


In [2]:
def list_checkpoint_files(batch_dir: Path = BATCH_DIR):
    files = sorted(
        [p for p in batch_dir.glob(PATTERN) if p.suffix in {'.parquet', '.pkl'}],
        key=lambda p: p.name
    )
    return files


def parse_batch_index(path: Path) -> int | None:
    m = re.search(r'results_batch_(\d+)', path.name)
    return int(m.group(1)) if m else None


checkpoint_files = list_checkpoint_files()
print(f'Found {len(checkpoint_files)} checkpoint files.')
for p in checkpoint_files[-10:]:
    print(' -', p.name)

Found 411 checkpoint files.
 - results_batch_00402.parquet
 - results_batch_00403.parquet
 - results_batch_00404.parquet
 - results_batch_00405.parquet
 - results_batch_00406.parquet
 - results_batch_00407.parquet
 - results_batch_00408.parquet
 - results_batch_00409.parquet
 - results_batch_00410.parquet
 - results_batch_00411.parquet


In [3]:
if checkpoint_files:
    summary_rows = []
    for p in checkpoint_files:
        stat = p.stat()
        summary_rows.append({
            'batch_index': parse_batch_index(p),
            'file': p.name,
            'format': p.suffix.replace('.', ''),
            'size_mb': round(stat.st_size / (1024 * 1024), 3),
            'modified_at': datetime.fromtimestamp(stat.st_mtime).strftime('%Y-%m-%d %H:%M:%S'),
        })

    files_df = pl.DataFrame(summary_rows).sort('batch_index')
    display(files_df)
else:
    files_df = pl.DataFrame()
    print('No checkpoint files found yet.')

files_df

batch_index,file,format,size_mb,modified_at
i64,str,str,f64,str
1,"""results_batch_00001.parquet""","""parquet""",2.504,"""2026-03-15 10:58:59"""
2,"""results_batch_00002.parquet""","""parquet""",4.934,"""2026-03-15 11:07:50"""
3,"""results_batch_00003.parquet""","""parquet""",3.294,"""2026-03-15 11:13:12"""
4,"""results_batch_00004.parquet""","""parquet""",6.887,"""2026-03-15 11:22:07"""
5,"""results_batch_00005.parquet""","""parquet""",4.626,"""2026-03-15 11:28:12"""
…,…,…,…,…
407,"""results_batch_00407.parquet""","""parquet""",0.719,"""2026-03-17 06:10:14"""
408,"""results_batch_00408.parquet""","""parquet""",0.74,"""2026-03-17 06:15:13"""
409,"""results_batch_00409.parquet""","""parquet""",0.757,"""2026-03-17 06:19:58"""


batch_index,file,format,size_mb,modified_at
i64,str,str,f64,str
1,"""results_batch_00001.parquet""","""parquet""",2.504,"""2026-03-15 10:58:59"""
2,"""results_batch_00002.parquet""","""parquet""",4.934,"""2026-03-15 11:07:50"""
3,"""results_batch_00003.parquet""","""parquet""",3.294,"""2026-03-15 11:13:12"""
4,"""results_batch_00004.parquet""","""parquet""",6.887,"""2026-03-15 11:22:07"""
5,"""results_batch_00005.parquet""","""parquet""",4.626,"""2026-03-15 11:28:12"""
…,…,…,…,…
407,"""results_batch_00407.parquet""","""parquet""",0.719,"""2026-03-17 06:10:14"""
408,"""results_batch_00408.parquet""","""parquet""",0.74,"""2026-03-17 06:15:13"""
409,"""results_batch_00409.parquet""","""parquet""",0.757,"""2026-03-17 06:19:58"""


In [4]:
def read_checkpoint(path: Path) -> pl.DataFrame:
    if path.suffix == '.parquet':
        return pl.read_parquet(path)
    if path.suffix == '.pkl':
        return pl.from_pandas(pd.read_pickle(path))
    return pl.DataFrame()


def load_checkpoints(files: list[Path], latest_n: int | None = None) -> pl.DataFrame:
    use_files = files[-latest_n:] if latest_n else files
    frames = []

    for p in use_files:
        try:
            df_part = read_checkpoint(p).with_columns(pl.lit(p.name).alias('_source_file'))
            frames.append(df_part)
        except Exception as e:
            print(f'Skipping unreadable file {p.name}: {e}')

    if not frames:
        return pl.DataFrame()

    return pl.concat(frames, how='diagonal_relaxed')


# Set latest_n=None to load everything, or an int (e.g. 5) for quicker live checks
latest_n = None
results_df = load_checkpoints(checkpoint_files, latest_n=latest_n)

print(f'Loaded rows: {results_df.height:,}')
print(f'Columns: {results_df.width}')
display(results_df.head(10))

Loaded rows: 411,000
Columns: 20


track_id,original_name,original_artist,id,name,title,uri,type,duration,duration_ms,artists,preview_url,is_explicit,is_playable,album,release_date,metadata_status,lyrics,lyrics_status,_source_file
str,str,str,str,str,str,str,str,f64,f64,list[struct[3]],str,bool,bool,struct[3],str,str,str,str,str
"""1IZc5f9ulw7Ge2eOpQH7w6""","""Kingdom Come""","""The Civil Wars""","""1IZc5f9ulw7Ge2eOpQH7w6""","""Kingdom Come""","""Kingdom Come""","""spotify:track:1IZc5f9ulw7Ge2eO…","""track""",222560.0,222560.0,"[{""6J7rw7NELJUCThPbAfyLIE"",""The Civil Wars"",""spotify:artist:6J7rw7NELJUCThPbAfyLIE""}]","""https://p.scdn.co/mp3-preview/…",false,true,"{[{300,""https://image-cdn-ak.spotifycdn.com/image/ab67616d00001e02d29a5d713277b0b3d517507b"",300}, {64,""https://image-cdn-ak.spotifycdn.com/image/ab67616d00004851d29a5d713277b0b3d517507b"",64}, {640,""https://image-cdn-ak.spotifycdn.com/image/ab67616d0000b273d29a5d713277b0b3d517507b"",640}],"""",""album""}","""2012-01-01""","""ok""","""Run, run, run away Buy yoursel…","""ok_lrclib""","""results_batch_00001.parquet"""
"""53QF56cjZA9RTuuMZDrSA6""","""I Won't Give Up""","""Jason Mraz""","""53QF56cjZA9RTuuMZDrSA6""","""I Won't Give Up""","""I Won't Give Up""","""spotify:track:53QF56cjZA9RTuuM…","""track""",240165.0,240165.0,"[{""4phGZZrJZRo4ElhRtViYdl"",""Jason Mraz"",""spotify:artist:4phGZZrJZRo4ElhRtViYdl""}]","""https://p.scdn.co/mp3-preview/…",false,true,"{[{300,""https://image-cdn-fa.spotifycdn.com/image/ab67616d00001e023f1ceb305415a090c0885986"",300}, {64,""https://image-cdn-fa.spotifycdn.com/image/ab67616d000048513f1ceb305415a090c0885986"",64}, {640,""https://image-cdn-fa.spotifycdn.com/image/ab67616d0000b2733f1ceb305415a090c0885986"",640}],"""",""album""}","""2012-04-13""","""ok""","""When I look into your eyes It'…","""ok_lrclib""","""results_batch_00001.parquet"""
"""7j6t0kr66wFLkBjjKxh5FJ""","""Watching You Watch Him""","""Eric Hutchinson""","""7j6t0kr66wFLkBjjKxh5FJ""","""Watching You Watch Him""","""Watching You Watch Him""","""spotify:track:7j6t0kr66wFLkBjj…","""track""",211026.0,211026.0,"[{""39x8gyJjTHiBQklFgVJSV4"",""Eric Hutchinson"",""spotify:artist:39x8gyJjTHiBQklFgVJSV4""}]","""https://p.scdn.co/mp3-preview/…",false,true,"{[{300,""https://image-cdn-ak.spotifycdn.com/image/ab67616d00001e028dc99119027de09032cde30a"",300}, {64,""https://image-cdn-ak.spotifycdn.com/image/ab67616d000048518dc99119027de09032cde30a"",64}, {640,""https://image-cdn-ak.spotifycdn.com/image/ab67616d0000b2738dc99119027de09032cde30a"",640}],"""",""album""}","""2012-04-13""","""ok""","""I love you from the bottom of …","""ok_lrclib""","""results_batch_00001.parquet"""
"""3Y6BuzQCg9p4yH347Nn8OW""","""Dancing Shoes""","""Green River Ordinance""","""3Y6BuzQCg9p4yH347Nn8OW""","""Dancing Shoes""","""Dancing Shoes""","""spotify:track:3Y6BuzQCg9p4yH34…","""track""",232373.0,232373.0,"[{""6Yuow6YoiBaVPFNjZ5BQi7"",""Green River Ordinance"",""spotify:artist:6Yuow6YoiBaVPFNjZ5BQi7""}]","""https://p.scdn.co/mp3-preview/…",false,true,"{[{300,""https://image-cdn-fa.spotifycdn.com/image/ab67616d00001e02afd0fff66836ee45dbbc0645"",300}, {64,""https://image-cdn-fa.spotifycdn.com/image/ab67616d00004851afd0fff66836ee45dbbc0645"",64}, {640,""https://image-cdn-fa.spotifycdn.com/image/ab67616d0000b273afd0fff66836ee45dbbc0645"",640}],"""",""album""}","""2012-02-28""","""ok""","""Put on your old black dress An…","""ok_lrclib""","""results_batch_00001.parquet"""
"""0J1iFISejodqBIy0Q9AS0N""","""I Shall Be Released""","""Eddie Vedder""","""0J1iFISejodqBIy0Q9AS0N""","""I Shall Be Released""","""I Shall Be Released""","""spotify:track:0J1iFISejodqBIy0…","""track""",283586.0,283586.0,"[{""0mXTJETA4XUa12MmmXxZJh"",""Eddie Vedder"",""spotify:artist:0mXTJETA4XUa12MmmXxZJh""}, {""3GBPw9NK25X1Wt2OUvOwY3"",""Jack Johnson"",""spotify:artist:3GBPw9NK25X1Wt2OUvOwY3""}, {""5HMFK1o5Bl9qY10PgavMhT"",""Zach Gill"",""spotify:artist:5HMFK1o5Bl9qY10PgavMhT""}]","""https://p.scdn.co/mp3-preview/…",false,true,"{[{300,""https://image-cdn-fa.spotifycdn.com/image

In [5]:
EXPECTED_BASE_COLS = [
    'track_id', 'original_name', 'original_artist',
    'metadata_status', 'lyrics', 'lyrics_status'
]

if results_df.is_empty():
    print('No loaded rows to inspect yet.')
else:
    missing = [c for c in EXPECTED_BASE_COLS if c not in results_df.columns]
    if missing:
        print('Missing expected extractor-v8 columns:', missing)
    else:
        print('Extractor-v8 base columns found.')

    if 'track_id' in results_df.columns:
        unique_tracks = results_df.select(pl.col('track_id').cast(pl.Utf8).n_unique()).item()
        duplicates = results_df.height - unique_tracks
        print(f'Unique track_id: {unique_tracks:,}')
        print(f'Duplicate rows by track_id: {duplicates:,}')

    if 'lyrics_status' in results_df.columns:
        status_counts = (
            results_df
            .with_columns(pl.col('lyrics_status').cast(pl.Utf8).fill_null('null').alias('lyrics_status'))
            .group_by('lyrics_status')
            .len()
            .sort('len', descending=True)
        )
        display(status_counts)

        print('\nFull status_counts (no truncation):')
        print('lyrics_status | len')
        print('-' * 80)
        for row in status_counts.iter_rows(named=True):
            print(f"{row['lyrics_status']} | {row['len']}")

        ok_count = results_df.filter(pl.col('lyrics_status').cast(pl.Utf8).str.contains('ok', literal=True)).height
        not_found_only = results_df.filter(
            pl.col('lyrics_status').cast(pl.Utf8).str.contains('not_found', literal=True)
            & ~pl.col('lyrics_status').cast(pl.Utf8).str.contains('ok', literal=True)
        ).height
        print(f'\n\nTotal "ok" count (including variants): {ok_count:,}')
        print(f'Total "not_found" count (without ok fallback): {not_found_only:,}')


Extractor-v8 base columns found.
Unique track_id: 411,000
Duplicate rows by track_id: 0


lyrics_status,len
str,u32
"""ok_lrclib""",257585
"""not_found_lrclib->error_librel…",51142
"""not_found_lrclib->error_librel…",38557
"""not_found_lrclib->error_librel…",21991
"""instrumental_lrclib->error_lib…",10203
…,…
"""not_found_lrclib->error_librel…",1
"""http_400_lrclib->error_librely…",1
"""not_found_lrclib->error_librel…",1



Full status_counts (no truncation):
lyrics_status | len
--------------------------------------------------------------------------------
ok_lrclib | 257585
not_found_lrclib->error_librelyrics: LyricsNotFound->bot_block_genius->not_found_synced | 51142
not_found_lrclib->error_librelyrics: LyricsNotFound->not_found_genius->not_found_synced | 38557
not_found_lrclib->error_librelyrics: LyricsNotFound->ok_genius | 21991
instrumental_lrclib->error_librelyrics: LyricsNotFound->not_found_genius->not_found_synced | 10203
instrumental_lrclib->error_librelyrics: LyricsNotFound->bot_block_genius->not_found_synced | 7178
not_found_lrclib->error_librelyrics: LyricsNotFound->bot_block_genius->ok_synced | 5697
not_found_lrclib->ok_librelyrics | 3793
instrumental_lrclib->error_librelyrics: LyricsNotFound->ok_genius | 3269
not_found_lrclib->skipped_no_auth->ok_genius | 3202
not_found_lrclib->skipped_no_auth->not_found_genius->not_found_synced | 2486
not_found_lrclib->error_librelyrics: LyricsNotFound->

In [6]:
# Quick check of the latest written checkpoint
if checkpoint_files:
    latest_file = checkpoint_files[-1]
    latest_df = read_checkpoint(latest_file)
    print(f'Latest file: {latest_file.name}')
    print(f'Rows in latest file: {latest_df.height:,}')
    print(f'Cols in latest file: {latest_df.width}')
    display(latest_df.head(10))

    if 'lyrics_status' in latest_df.columns:
        latest_status = latest_df.group_by('lyrics_status').len().sort('len', descending=True)
        display(latest_status)
else:
    print('No checkpoint files found.')

Latest file: results_batch_00411.parquet
Rows in latest file: 1,000
Cols in latest file: 19


track_id,original_name,original_artist,id,name,title,uri,type,duration,duration_ms,artists,preview_url,is_explicit,is_playable,album,release_date,metadata_status,lyrics,lyrics_status
str,str,str,str,str,str,str,str,i64,i64,list[struct[3]],str,bool,bool,struct[3],str,str,str,str
"""3HCw9YPoNxyFdjDNW4PNpU""","""Más Que Nada""","""Jessy J""","""3HCw9YPoNxyFdjDNW4PNpU""","""Más Que Nada""","""Más Que Nada""","""spotify:track:3HCw9YPoNxyFdjDN…","""track""",304146,304146,"[{""4WrtIP5PIekZwaAZo1tb0x"",""Jessy J"",""spotify:artist:4WrtIP5PIekZwaAZo1tb0x""}]","""https://p.scdn.co/mp3-preview/…",false,true,"{[{300,""https://image-cdn-fa.spotifycdn.com/image/ab67616d00001e02104bf24adcc5e2eb148d8d8b"",300}, {64,""https://image-cdn-fa.spotifycdn.com/image/ab67616d00004851104bf24adcc5e2eb148d8d8b"",64}, {640,""https://image-cdn-fa.spotifycdn.com/image/ab67616d0000b273104bf24adcc5e2eb148d8d8b"",640}],"""",""album""}","""2008-03-04""","""ok""","""Mas que nada Sai da minha fren…","""ok_lrclib"""
"""2FIEhpjfQFHn2Tjl5ubfV0""","""Pennant""","""Little Symphony""","""2FIEhpjfQFHn2Tjl5ubfV0""","""Pennant""","""Pennant""","""spotify:track:2FIEhpjfQFHn2Tjl…","""track""",109030,109030,"[{""4SCWiQbJCMTHK737aNUqBJ"",""Little Symphony"",""spotify:artist:4SCWiQbJCMTHK737aNUqBJ""}]","""https://p.scdn.co/mp3-preview/…",false,true,"{[{300,""https://image-cdn-fa.spotifycdn.com/image/ab67616d00001e02922ee736c9de99e6332fbf22"",300}, {64,""https://image-cdn-fa.spotifycdn.com/image/ab67616d00004851922ee736c9de99e6332fbf22"",64}, {640,""https://image-cdn-fa.spotifycdn.com/image/ab67616d0000b273922ee736c9de99e6332fbf22"",640}],"""",""album""}","""2021-07-23""","""ok""",null,"""not_found_lrclib->error_librel…"
"""472ybTittytWeHFhL8hsR6""","""Wet Mulch""","""Equipment""","""472ybTittytWeHFhL8hsR6""","""Wet Mulch""","""Wet Mulch""","""spotify:track:472ybTittytWeHFh…","""track""",264869,264869,"[{""1xxn3mhlUmOugl1ZhE0Mcx"",""Equipment"",""spotify:artist:1xxn3mhlUmOugl1ZhE0Mcx""}]","""https://p.scdn.co/mp3-preview/…",false,true,"{[{300,""https://image-cdn-fa.spotifycdn.com/image/ab67616d00001e02e0ef66ff752b643de06c071d"",300}, {64,""https://image-cdn-fa.spotifycdn.com/image/ab67616d00004851e0ef66ff752b643de06c071d"",64}, {640,""https://image-cdn-fa.spotifycdn.com/image/ab67616d0000b273e0ef66ff752b643de06c071d"",640}],"""",""album""}","""2019-10-25""","""ok""","""This is the one No more slack…","""ok_lrclib"""
"""47xaObKJyJPGncrgz4Fo6M""","""Fallen Into Disuse""","""Wormrot""","""47xaObKJyJPGncrgz4Fo6M""","""Fallen Into Disuse""","""Fallen Into Disuse""","""spotify:track:47xaObKJyJPGncrg…","""track""",79626,79626,"[{""3vMnvW7u5207ATyxTQIxNz"",""Wormrot"",""spotify:artist:3vMnvW7u5207ATyxTQIxNz""}]","""https://p.scdn.co/mp3-preview/…",true,true,"{[{300,""https://image-cdn-fa.spotifycdn.com/image/ab67616d00001e0226754b34e5949bc7316a4339"",300}, {64,""https://image-cdn-fa.spotifycdn.com/image/ab67616d0000485126754b34e5949bc7316a4339"",64}, {640,""https://image-cdn-fa.spotifycdn.com/image/ab67616d0000b27326754b34e5949bc7316a4339"",640}],"""",""album""}","""2016-10-14""","""ok""","""Yeah! Clouded by reality Horr…","""ok_lrclib"""
"""7at9R8GFWtbh4dgSOFL1Sr""","""Dark Green Water""","""Great Grandpa""","""7at9R8GFWtbh4dgSOFL1Sr""","""Dark Green Water""","""Dark Green Water""","""spotify:track:7at9R8GFWtbh4dgS…","""track""",253840,253840,"[{""1Hs5RG6WIwUSJLxRYWaOW6"",""Great Grandpa"",""spotify:artist:1Hs5RG6WIwUSJLxRYWaOW6""}]","""https://p.scdn.co/mp3-preview/…",false,true,"{[{300,""https://image-cdn-fa.spotifycdn.com/image/ab67616d00001e02edd910d6d667fb53fd9cb1f7"",300}, {64,""https://image-cdn-fa.spotifycdn.com/image/ab67616d00004851edd910d6d667fb53fd9cb1f7"",64}, {640,""https://image-cdn-fa.spotifycdn.com/image/ab67616d0000b273edd910d6d667fb53fd9cb1f7"",640}],"""",""album""}","""2019-10-25""","""ok""","""Dark Green Water All things fa…","""ok_lrclib"""
"""4vil6lFUWmKimuuK0tDxFP""","""Happy Little Boozer""","""Korpiklaani""","""4vil6lFUWmKimuuK0tDxFP""","""Happy Little Boozer

lyrics_status,len
str,u32
"""ok_lrclib""",664
"""not_found_lrclib->error_librel…",134
"""not_found_lrclib->error_librel…",112
"""instrumental_lrclib->error_lib…",40
"""instrumental_lrclib->error_lib…",24
"""not_found_lrclib->error_librel…",14
"""not_found_lrclib->error_librel…",7
"""not_found_lrclib->ok_librelyri…",4
"""instrumental_lrclib->error_lib…",1


In [7]:
# Optional live monitor: set rounds > 1 to auto-refresh
rounds = 1           # e.g. 20
sleep_seconds = 10   # e.g. 10

for i in range(1, rounds + 1):
    files_now = list_checkpoint_files()
    latest_name = files_now[-1].name if files_now else 'none'
    print(f'[{i}/{rounds}] files={len(files_now)} latest={latest_name}')

    if i < rounds:
        time.sleep(sleep_seconds)

[1/1] files=411 latest=results_batch_00411.parquet
